# Convert NetCDF To Zarr

This notebook converts the ice-temperature NetCDF example into a consolidated Zarr store. Zarr is useful for multidimensional arrays because chunked object storage lets clients read selected depths, windows, or variables without downloading one monolithic file.


In [4]:
from pathlib import Path


def find_repo_root(start: Path = Path.cwd()) -> Path:
    """Find the repository root when the kernel starts in a subfolder."""
    start = start.resolve()
    for path in (start, *start.parents):
        if (path / ".git").exists() or (path / "downloaded_data").exists():
            return path
    return start


REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "downloaded_data"
DATA_DIR.mkdir(exist_ok=True)

import numpy as np
import rioxarray
import xarray as xr

NC_FILE = DATA_DIR / "SM_TEST_MIR_ITUDP4_20130101T000000_20141231T000000_200_001_0.nc"
ZARR_STORE = DATA_DIR / "SM_TEST_MIR_ITUDP4_20130101T000000_20141231T000000_200_001_0.zarr"
CHUNKS = {'depth': 2, 'y':5063, 'x':5673}


## Open With Explicit Chunks And CRS Metadata


In [5]:
ds = xr.open_dataset(NC_FILE, chunks=CHUNKS, decode_coords="all")
source_crs = ds.attrs.get("srid")
if source_crs:
    ds = ds.rio.write_crs(source_crs)
    if {"x", "y"}.issubset(ds.dims):
        ds = ds.rio.set_spatial_dims(x_dim="x", y_dim="y")
        ds = ds.rio.write_coordinate_system()
ds

<xarray.Dataset> Size: 17MB
Dimensions:      (x: 200, y: 224, depth: 91)
Coordinates:
  * depth        (depth) int16 182B 0 50 100 150 200 ... 4350 4400 4450 4500
    spatial_ref  int64 8B 0
Dimensions without coordinates: x, y
Data variables:
    latitude     (x, y) float64 358kB dask.array<chunksize=(200, 224), meta=np.ndarray>
    longitude    (x, y) float64 358kB dask.array<chunksize=(200, 224), meta=np.ndarray>
    Tice         (x, y, depth) float32 16MB dask.array<chunksize=(200, 224, 2), meta=np.ndarray>
    QualityFlag  (x, y) float32 179kB dask.array<chunksize=(200, 224), meta=np.ndarray>
Attributes: (12/21)
    product:               Internal ice sheet temperature
    product_version:       2.0
    institution:           IFAC-CNR, IGE
    description:           developed from SMOS L3 (CATDS)
    source:                IFAC L4Tice processor
    project:               ESA CryoSMOS project (contract n.4000112262/14/I-NB)
    ...                    ...
    geospatial_lat_max:    0
    geospatial_lon_min:    -180
    geospatial_lon_max:    180
    spatial_resolution:    25 km
    srid:                  EPSG:6932
    creation_date:         2019-09-20T15:14:23

## Write Consolidated Zarr

Consolidated metadata stores array metadata in one place, which avoids many small metadata reads from object storage.


In [7]:
write_kwargs = dict(mode="w", consolidated=True)
ds.to_zarr(ZARR_STORE, zarr_format=2, write_empty_chunks=False, **write_kwargs)